# DS2002 · Walmart Warm-Up SKU Mess

**Lab — 2026-10-16 · Fall 2026**  

---

## Lab 08 — Walmart Warm-Up (SKU Mess)

This lab uses the **real midterm data** in `../06-midterm-walmart/data/`. It's a warm-up so you hit the ground running on the project. On Kaggle/Colab, upload the data folder or adjust the path.

*If the file isn't found, the cell falls back to a small inline sample so the notebook still runs.*

In [34]:
import pandas as pd
PATH = '/06-midterm-walmart/data/product_catalog.csv'
try:
    cat = pd.read_csv(PATH)
except FileNotFoundError:
    from io import StringIO
    cat = pd.read_csv(StringIO('sku,product_name,category\nPT-12,pop-tarts strawberry,Breakfast Foods\nSKU#459812,Strawberry Pop-Tart,Breakfast\nPOPTART-STRAW,Pop-Tarts Strawberry,Breakfast'))
cat.head(10)

,sku,product_name,category
0,PT-12,pop-tarts strawberry,Breakfast Foods
1,SKU#459812,Strawberry Pop-Tart,Breakfast
2,POPTART-STRAW,Pop-Tarts Strawberry,Breakfast


In [35]:
import os
print(os.getcwd())
print(os.path.exists(PATH))

/content
False


### Q1 — How many rows look like Pop-Tart products?

Filter on `product_name` containing `pop`, case-insensitively. Print the count and the matching rows.

In [26]:
# TODO (hint: .str.lower().str.contains('pop'))
pop = cat[cat['product_name'].str.lower().str.contains('pop', na=False)]
print(len(pop))
pop

3


,sku,product_name,category
0,PT-12,pop-tarts strawberry,Breakfast Foods
1,SKU#459812,Strawberry Pop-Tart,Breakfast
2,POPTART-STRAW,Pop-Tarts Strawberry,Breakfast


### Q2 — Which SKUs and names all mean *strawberry* Pop-Tart?

List them. Then note which Pop-Tart rows are **not** strawberry — those must stay separate, and saying so is part of the answer.

In [27]:
# TODO
is_straw = pop['product_name'].str.lower().str.contains('straw', na=False)
straw = pop[is_straw]
not_straw = pop[~is_straw]
print(straw[['sku', 'product_name']])
print(not_straw[['sku', 'product_name']])

             sku          product_name
0          PT-12  pop-tarts strawberry
1     SKU#459812   Strawberry Pop-Tart
2  POPTART-STRAW  Pop-Tarts Strawberry
Empty DataFrame
Columns: [sku, product_name]
Index: []


**Pop-Tart rows that are a different product:** _..._

### Q3 — Build the mapping

A dict from each variant SKU to one canonical SKU. Then apply it in a way that leaves unmapped SKUs as themselves rather than turning them into `NaN` — that single mistake silently deletes most of the catalog.

In [28]:
# TODO
CANON = 'POPTART-STRAW'   # pick one; the most descriptive SKU is a fine choice
mapping = {sku: CANON for sku in straw['sku'] if sku != CANON}

cat['sku_canon'] = cat['sku'].map(mapping).fillna(cat['sku'])

print(mapping)
cat[['sku', 'sku_canon', 'product_name']]

{'PT-12': 'POPTART-STRAW', 'SKU#459812': 'POPTART-STRAW'}


,sku,sku_canon,product_name
0,PT-12,POPTART-STRAW,pop-tarts strawberry
1,SKU#459812,POPTART-STRAW,Strawberry Pop-Tart
2,POPTART-STRAW,POPTART-STRAW,Pop-Tarts Strawberry


### Q4 — Normalize `category`

Strip and title-case, then show the distinct values before and after with counts. Note any pair that is still two labels for one thing — `Breakfast` and `Breakfast Foods` are not the same string but they are the same category.

In [29]:
# TODO
print(cat['category'].value_counts(dropna=False))   # before

cat['category'] = cat['category'].str.strip().str.title()
print(cat['category'].value_counts(dropna=False))   # after strip/title

cat['category'] = cat['category'].replace({'Breakfast Foods': 'Breakfast'})
print(cat['category'].value_counts(dropna=False))   # after merging synonyms

category
Breakfast          2
Breakfast Foods    1
Name: count, dtype: int64
category
Breakfast          2
Breakfast Foods    1
Name: count, dtype: int64
category
Breakfast    3
Name: count, dtype: int64


### Q5 — Find the fragmentation you have not spotted yet

Hand-scanning a catalog does not scale. **TODO:** build a candidate list programmatically — normalize each product name (lowercase, strip punctuation, sort the words) and group by that, then show any group holding more than one SKU.

This is the workflow from Monday's lecture, and it is how you will actually approach the full catalog on the midterm.

In [30]:
# TODO
import re

def name_key(s):
    s = str(s).lower()
    s = re.sub(r'[^a-z0-9\s]', ' ', s)   # punctuation -> space
    return ' '.join(sorted(s.split()))

cat['name_key'] = cat['product_name'].map(name_key)

groups = (cat.groupby('name_key')['sku']
             .agg(['nunique', lambda x: sorted(x.unique())])
             .rename(columns={'<lambda_0>': 'skus'}))
candidates = groups[groups['nunique'] > 1].sort_values('nunique', ascending=False)
candidates


,nunique,skus
name_key,,
pop strawberry tarts,2,"[POPTART-STRAW, PT-12]"


### Q6 — Prove the mapping is safe

**TODO:** assert that applying your mapping produced no nulls in the canonical column and did not change the number of rows. Consolidation renames; it never adds or removes.

In [32]:
before = cat.copy()
after = before.assign(sku_canon=before['sku'].map(mapping).fillna(before['sku']))

assert after['sku_canon'].notna().all(), "mapping produced nulls"
assert len(after) == len(before), "row count changed"
unmapped = ~before['sku'].isin(mapping.keys())
assert (after.loc[unmapped, 'sku_canon'] == before.loc[unmapped, 'sku']).all()

print("mapping safe:", len(after), "rows,",
      before['sku'].nunique(), "->", after['sku_canon'].nunique(), "distinct SKUs")

mapping safe: 3 rows, 3 -> 1 distinct SKUs


### Write-up

**a)** Name two product families beyond Pop-Tarts that appear fragmented, and how you would group them.



**b)** Where does consolidation stop being a data-cleaning task and become a business decision? Give one example from this catalog where you genuinely are not sure, and say who you would ask.

A) I couldn't get the data set to load without my collab tweaking out
B) Data consolidation becomes a business decision when a good goes from having a singular option to multiple options under the same category, for instance multiple options in the size of the packaging, it would be good for knowing aggregate sales of an item but completely useless for stocking and shelving of specific sized packages of items.